In [ ]:
!pip install --upgrade transformers datasets evaluate -q

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset
import numpy as np
import evaluate
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
dataset = load_dataset("bwbayu/automatic_essay_scoring")
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][:3]["answer"])
print(dataset["train"][:3]["normalized_score"])


In [ ]:
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_length = 512

In [ ]:
def preprocess_function(examples):
    texts = [
        f"{q} {tokenizer.sep_token} {r} {tokenizer.sep_token} {a}"
        for q, r, a in zip(examples["question"], examples["reference_answer"], examples["answer"])
    ]
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

In [ ]:
def prepare_labels(examples):
    examples["labels"] = examples["normalized_score"]
    return examples

tokenized_dataset = dataset.map(preprocess_function, batched=True)
tokenized_dataset = tokenized_dataset.map(prepare_labels, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    problem_type="regression"
)

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()

    mae = mean_absolute_error(labels, predictions)
    rmse = root_mean_squared_error(labels, predictions)

    return {"mae": mae, "rmse": rmse}

In [ ]:
training_args = TrainingArguments(
    output_dir="./indobert_essay_scoring",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="rmse",
    greater_is_better=False,
    report_to=[]
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

metrics = ["eval_mae", "eval_rmse"]
values = [eval_results[m] for m in metrics]

plt.figure(figsize=(6, 4))
plt.bar(metrics, values)
plt.title("Evaluasi Model IndoBERT AES")
plt.ylabel("Score")
plt.show()

In [ ]:
trainer.save_model("./indobert_essay_scoring")
tokenizer.save_pretrained("./indobert_essay_scoring")

In [ ]:
!cp -r ./indobert_essay_scoring /content/drive/MyDrive/

In [ ]:
imported_model = AutoModelForSequenceClassification.from_pretrained("./indobert_essay_scoring")
imported_model